In [1]:
# Install: %pip install pandas numpy matplotlib scikit-learn vaderSentiment

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Load dataset
df = pd.read_csv("Atlantic_Sapphire_vs_Sanford_Stock_Comparison.csv")

# K-Means clustering
num = df.select_dtypes(include="number").dropna(axis=1, how="all")
num = num.loc[:, num.nunique() > 1]

if len(df) >= 3 and num.shape[1] >= 2:
    X = SimpleImputer(strategy="median").fit_transform(num)
    X = StandardScaler().fit_transform(X)
    df["Cluster"] = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X)
    plt.scatter(X[:, 0], X[:, 1], c=df["Cluster"])
    plt.title("Atlantic Sapphire vs Sanford: K-Means")
    plt.show()

# NLP and sentiment analysis
text_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

if text_cols:
    text = df[text_cols].fillna("").astype(str).agg(" ".join, axis=1)
    valid = text.str.strip().ne("")

    if valid.sum() >= 2:
        tfidf = TfidfVectorizer(stop_words="english", max_features=1000)
        matrix = tfidf.fit_transform(text[valid])
        print("Top NLP terms:", tfidf.get_feature_names_out()[
            matrix.sum(axis=0).A1.argsort()[::-1][:10]
        ])

        analyzer = SentimentIntensityAnalyzer()
        df["Sentiment"] = float("nan")
        df.loc[valid, "Sentiment"] = text[valid].apply(
            lambda t: analyzer.polarity_scores(t)["compound"]
        )
else:
    print("NLP requires text columns such as financial news or commentary.")

# Save results
df.to_csv("Atlantic_Sapphire_Sanford_Results.csv", index=False)
print("Analysis complete. Results saved.")

ModuleNotFoundError: No module named 'vaderSentiment'